# 04 — Training Visualization: Loss and Accuracy over 50 Epochs

**Objective:** Train a Keras neural network on MNIST for **50 epochs** and visualize how training and validation loss and accuracy evolve over the full training run.
This demonstrates learning progress, convergence behaviour, and early signs of overfitting.

---
## 1. Concepts

### What do training curves tell us?

- **Decreasing training loss** — the model is fitting the training data.
- **Decreasing validation loss** — the model is generalizing to unseen data.
- **Diverging training vs validation loss** — the model has started to **overfit**: it memorizes training examples but fails to generalize.
- **Plateau** — learning has stalled; reducing the learning rate or applying regularization may help.

50 epochs is long enough to observe both the initial fast-learning phase and the later plateau/overfitting phase.

---
## 2. Imports and Configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# --- Training configuration ---
EPOCHS     = 50      # Required: 50 epochs
BATCH_SIZE = 128
VAL_SPLIT  = 0.1
HIDDEN_1   = 128
HIDDEN_2   = 64
# ------------------------------

print(f"TensorFlow : {tf.__version__}")
print(f"Training for {EPOCHS} epochs.")

---
## 3. Load and Preprocess MNIST

In [ ]:
(X_train_raw, y_train), (X_test_raw, y_test) = keras.datasets.mnist.load_data()

# Normalize to [0, 1]
X_train = X_train_raw.astype("float32") / 255.0
X_test  = X_test_raw.astype("float32")  / 255.0

print(f"Training set : {X_train.shape}")
print(f"Test set     : {X_test.shape}")

---
## 4. Build and Compile Model

In [ ]:
model = keras.Sequential([
    keras.layers.Flatten(input_shape=(28, 28), name="flatten"),
    keras.layers.Dense(HIDDEN_1, activation="relu", name="hidden_1"),
    keras.layers.Dense(HIDDEN_2, activation="relu", name="hidden_2"),
    keras.layers.Dense(10, activation="softmax",   name="output"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

---
## 5. Train for 50 Epochs

> **Note:** No EarlyStopping is used here so all 50 epochs are always completed.
> This is intentional — the goal is to observe the full training curve.

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VAL_SPLIT,
    verbose=1
)

print(f"\nTraining complete. Ran for {len(history.history['loss'])} epochs.")

---
## 6. Test Set Evaluation

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Final Test Loss     : {test_loss:.4f}")
print(f"Final Test Accuracy : {test_acc * 100:.2f}%")

---
## 7. Visualize Loss over 50 Epochs

In [ ]:
epochs_range = range(1, EPOCHS + 1)
train_loss   = history.history["loss"]
val_loss     = history.history["val_loss"]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(epochs_range, train_loss, label="Training Loss",   color="steelblue", linewidth=2)
ax.plot(epochs_range, val_loss,   label="Validation Loss", color="tomato",    linewidth=2, linestyle="--")
ax.set_title("Training vs Validation Loss over 50 Epochs", fontsize=14)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss (Sparse Categorical Crossentropy)")
ax.legend(fontsize=12)
ax.grid(True, linestyle="--", alpha=0.5)
# Mark best validation loss
best_epoch = int(np.argmin(val_loss)) + 1
ax.axvline(best_epoch, color="gray", linestyle=":", linewidth=1.5,
           label=f"Best val loss @ epoch {best_epoch}")
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print(f"Best validation loss at epoch {best_epoch}: {val_loss[best_epoch-1]:.4f}")

---
## 8. Visualize Accuracy over 50 Epochs

In [ ]:
train_acc = history.history["accuracy"]
val_acc   = history.history["val_accuracy"]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(epochs_range, [a * 100 for a in train_acc], label="Training Accuracy",   color="steelblue", linewidth=2)
ax.plot(epochs_range, [a * 100 for a in val_acc],   label="Validation Accuracy", color="tomato",    linewidth=2, linestyle="--")
ax.set_title("Training vs Validation Accuracy over 50 Epochs", fontsize=14)
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy (%)")
ax.set_ylim([min(min(train_acc), min(val_acc)) * 100 - 2, 101])
ax.legend(fontsize=12)
ax.grid(True, linestyle="--", alpha=0.5)

# Mark best validation accuracy
best_val_epoch = int(np.argmax(val_acc)) + 1
ax.axvline(best_val_epoch, color="gray", linestyle=":", linewidth=1.5,
           label=f"Best val acc @ epoch {best_val_epoch}")
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print(f"Best validation accuracy at epoch {best_val_epoch}: {val_acc[best_val_epoch-1] * 100:.2f}%")
print(f"Final training  accuracy (epoch {EPOCHS}): {train_acc[-1] * 100:.2f}%")
print(f"Final validation accuracy (epoch {EPOCHS}): {val_acc[-1] * 100:.2f}%")

---
## 9. Combined Summary Plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss
ax1.plot(epochs_range, train_loss, label="Train",      color="steelblue", linewidth=2)
ax1.plot(epochs_range, val_loss,   label="Validation", color="tomato",    linewidth=2, linestyle="--")
ax1.set_title("Loss", fontsize=13)
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.legend(); ax1.grid(True, linestyle="--", alpha=0.4)

# Accuracy
ax2.plot(epochs_range, [a * 100 for a in train_acc], label="Train",      color="steelblue", linewidth=2)
ax2.plot(epochs_range, [a * 100 for a in val_acc],   label="Validation", color="tomato",    linewidth=2, linestyle="--")
ax2.set_title("Accuracy", fontsize=13)
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)")
ax2.legend(); ax2.grid(True, linestyle="--", alpha=0.4)

fig.suptitle(f"MNIST Training Progress — 50 Epochs   |   Test Accuracy: {test_acc*100:.2f}%",
             fontsize=14)
plt.tight_layout()
plt.show()

---
## 10. Observations and Conclusion

### What the curves reveal:

1. **Fast initial improvement (epochs 1–10):** Training and validation loss both drop sharply. The network is learning quickly from the most discriminative features.
2. **Slowing convergence (epochs 10–30):** Improvements become smaller each epoch. The easy patterns are already learned.
3. **Possible overfitting (epochs 30+):** If validation loss increases while training loss continues to decrease, this confirms overfitting. With MNIST and no regularization, a mild divergence is common.

### Divergence Interpretation

- If `validation loss > training loss` and the gap widens: **overfitting** — the model is memorizing training examples.
- Remedies include: Dropout layers, L2 regularization, reducing model size, or EarlyStopping.

### Final Results

| Metric | Value |
|--------|-------|
| Epochs trained | 50 |
| Test accuracy | (see cell 6 output) |
| Best val-acc epoch | (see cell 8 output) |